# MDM2 GNN Pipeline — Colab (T4 GPU)

End-to-end: rank-fusion re-score → calibration audit → pIC50 inference → MDM2-like pretrain pool.
Assumes this notebook lives in `Part_17/` of the `new_method` repo.

| Step | Time | GPU? |
|---|---|---|
| 1-3 setup | ~3 min | — |
| 4 re-score (154k) | ~1 min | no |
| 5 calibration audit | ~5 min | no |
| 6 pIC50 demo (1k) | ~3 min | yes |
| 7 pIC50 full (154k) | ~15 min | yes |
| 8 pretrain pool | ~20-30 min | no |


In [ ]:
# 1) GPU check
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 2) Dependencies (re-run + Restart session if rdkit import fails after install)
!pip install -q "torch-geometric>=2.5" "rdkit>=2023.09" scikit-learn pandas matplotlib tqdm
import rdkit, sklearn, torch_geometric
print('rdkit', rdkit.__version__)

In [ ]:
# 3) Get repo (clone once, pull afterwards) + go to root
import os
if not os.path.isdir('new_method'):
    !git clone https://github.com/Techbjd/new_method.git
%cd new_method
!git pull --ff-only
!git log --oneline -3
!ls Part_17/scripts/

In [ ]:
# 4) Verify tracked inputs exist (all in repo, no manual upload needed)
import os
need = [
 'Part_17/results/coconut_gnn_screening_full.csv',
 'Part_17/results/coconut_gnn_screening_full_calibrated.csv',
 'Part_17/results/coconut_with_domain.csv',
 'Part_8/screening_results.csv',
 'Part_9/paper_rf116_filtered_compounds.csv',
 'Part_17/models/hybrid_pretrain_regression.pth',
 'Part_17/models/hybrid_scaffold_pretrain_r0.5.pth',
 'Part_17/models/rf_scaffold.pkl',
 'Part_17/data/featurized_curated.npz',
 'Part_17/data/mdm2_chembl_smiles.csv',
 'ro5_properties_filtered.csv',
]
missing = [p for p in need if not os.path.exists(p)]
print('missing:', missing if missing else 'NONE — all inputs present')

In [ ]:
# 5) Rank-fusion re-score (CPU, ~1 min) -> consensus_T1.csv (101 rows, vs old T1=1)
!python3 Part_17/scripts/rescore_rankfusion.py
!ls -lh Part_17/results/coconut_rescore_rankfusion.csv Part_17/results/top116_rankfusion.csv Part_17/results/consensus_T1.csv

In [ ]:
# 6) Calibration + overconfidence audit (CPU, ~5 min)
!python3 Part_17/scripts/audit_cal.py
!python3 Part_17/scripts/gnn_ece2.py
!python3 Part_17/scripts/finalize.py
!ls -lh Part_17/results/calibration_audit.csv Part_17/results/reliability_table.csv Part_17/results/coconut_disagreement_flags.csv

In [ ]:
# 7) pIC50 demo on 1k COCONUT (GPU, ~3 min) — proves pipeline
!python3 Part_17/scripts/run_pic50_demo.py --nrows 1000 --out Part_17/results/sample_pIC50_demo.csv
!head -5 Part_17/results/sample_pIC50_demo.csv

In [ ]:
# 8) FULL 154k pIC50 (GPU T4 ~15 min). Run this cell, then re-rank S3.
!python3 Part_17/scripts/run_pic50_demo.py --nrows 1000000 --out Part_17/results/coconut_full_pIC50.csv
!ls -lh Part_17/results/coconut_full_pIC50.csv

In [ ]:
# 9) OPTIONAL: rebuild MDM2-like pretrain pool (CPU ~20-30 min, downloads ZINC 22MB).
# Skip — Part_17/data/pretrain_mdm2like_20k.csv (2,913 rows) is already in the repo.
# !python3 Part_17/scripts/build_pretrain_pool.py

In [ ]:
# 10) Show results
import pandas as pd
print('--- top10 rank-fusion ---')
print(pd.read_csv('Part_17/results/top116_rankfusion.csv').head(10).to_string(index=False))
print()
print('--- consensus_T1 head ---')
print(pd.read_csv('Part_17/results/consensus_T1.csv').head(10).to_string(index=False))